Step 1: Import Libraries

Explanation:
• Import necessary libraries for data processing, text cleaning, sentiment analysis, and classification.
• NLTK provides tools like SentimentIntensityAnalyzer (VADER) for sentiment analysis and stopwords for text preprocessing.

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
import nltk
import re

# Download necessary NLTK data
nltk.download('vader_lexicon')
nltk.download('stopwords')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/guyparsadanov/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/guyparsadanov/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Step 2: Define a Preprocessing Function

Explanation:
• This function cleans the text by:
• Removing special characters, numbers, and punctuations.
• Converting all text to lowercase for uniformity.
• Removing common stopwords (e.g., “and”, “the”) that don’t contribute much to sentiment.

In [13]:
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    # Remove special characters, numbers, and punctuations
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Remove stopwords
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

Step 3: Load and Clean the Dataset

Explanation:
• Load the dataset containing reviews.
• Drop rows where reviews.text is missing to ensure only valid data is processed.
• Apply the preprocessing function to clean the review text.

In [14]:
# Load the dataset
data = pd.read_csv('1429_1.csv', delimiter=';')

# Keep relevant columns and drop rows with missing text data
data_cleaned = data[['reviews.text']].dropna()

# Preprocess the text data
data_cleaned['cleaned_text'] = data_cleaned['reviews.text'].apply(preprocess_text)

/var/folders/0v/2jlpljqd1msbtz5s_xyw4yj00000gn/T/ipykernel_97132/3328228912.py:2: DtypeWarning: Columns (21,24,25,26,27,28,29,30,31,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('1429_1.csv', delimiter=';')


Step 4: Sentiment Labeling with VADER

Explanation:
•	Use SentimentIntensityAnalyzer (VADER) to calculate the sentiment polarity of each review.
•	Define thresholds for labeling:
•	Positive: Compound polarity score > 0.2.
•	Negative: Compound polarity score < -0.2.
•	Neutral: Compound polarity score between -0.2 and 0.2.

In [15]:
# Use VADER for sentiment labeling
sia = SentimentIntensityAnalyzer()
def classify_sentiment_vader(text):
    sentiment_score = sia.polarity_scores(text)
    if sentiment_score['compound'] > 0.2:
        return "positive"
    elif sentiment_score['compound'] < -0.2:
        return "negative"
    else:
        return "neutral"

# Apply VADER sentiment analysis
data_cleaned['sentiment'] = data_cleaned['cleaned_text'].apply(classify_sentiment_vader)

In [16]:
# Use VADER for sentiment labeling
sia = SentimentIntensityAnalyzer()
def classify_sentiment_vader(text):
    sentiment_score = sia.polarity_scores(text)
    if sentiment_score['compound'] > 0.2:
        return "positive"
    elif sentiment_score['compound'] < -0.2:
        return "negative"
    else:
        return "neutral"

#classify_sentiment_vader('I am not sure about product!')

'negative'

Step 5: Split the Data

Explanation:
•	Divide the data into:
•	Features (X): The cleaned text reviews.
•	Labels (y): The corresponding sentiment (positive, neutral, negative).
•	Split into training and testing sets:
•	Training set (80%): Used to train the model.
•	Testing set (20%): Used to evaluate the model.

In [17]:
# Separate features (X) and labels (y)
X = data_cleaned['cleaned_text']
y = data_cleaned['sentiment']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
y

0        positive
1        positive
2        positive
3        positive
4        positive
           ...   
41415    positive
41416    positive
41417    positive
41418    positive
41419    negative
Name: sentiment, Length: 34441, dtype: object

Step 6: Vectorize the Text Data

Explanation:
•	Convert text into numerical format using TF-IDF (Term Frequency-Inverse Document Frequency).
•	This ensures the model understands the importance of words in each review.
•	Use max_features=10000 to capture up to 10,000 important words.

In [19]:
# Vectorize the text data using TF-IDF
vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)  # Fit and transform training data
X_test_tfidf = vectorizer.transform(X_test)        # Transform test data

Step 7: Train a Logistic Regression Model

Explanation:
•	Train a Logistic Regression model, which is robust for text classification tasks.
•	Use max_iter=500 to ensure the model converges during training.

In [24]:
# Train a Logistic Regression model
model = LogisticRegression(max_iter=500)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=500)

Step 8: Evaluate the Model

Explanation:
•	Predict sentiments on the test data.
•	Calculate:
•	Accuracy: Percentage of correct predictions.
•	Classification Report: Detailed metrics (precision, recall, F1-score) for each class.

In [25]:
# Evaluate the model
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Classification Report:\n", report)

Model Accuracy: 0.9203077369719843
Classification Report:
               precision    recall  f1-score   support

    negative       0.69      0.14      0.23       241
     neutral       0.89      0.84      0.87      1729
    positive       0.93      0.99      0.96      4919

    accuracy                           0.92      6889
   macro avg       0.84      0.66      0.69      6889
weighted avg       0.91      0.92      0.91      6889



In [22]:
import pandas as pd

def save_clustered_data_to_csv(data_cleaned, file_name):
    """
    Save the cleaned and sentiment-labeled data to a CSV file for inspection.

    Args:
    - data_cleaned (pd.DataFrame): The DataFrame containing cleaned text and sentiment labels.
    - file_name (str): The name of the output CSV file.

    Returns:
    - None
    """
    # Select only relevant columns for output
    output_data = data_cleaned[['reviews.text', 'cleaned_text', 'sentiment']]
    
    # Save to CSV file
    output_data.to_csv(file_name, index=False, sep=';', encoding='utf-8')
    print(f"Clustered data saved to {file_name}")


# Call the function after sentiment classification
save_clustered_data_to_csv(data_cleaned, 'clustered_sentiment_data.csv')

Clustered data saved to clustered_sentiment_data.csv
